## PHF-Crop mainline visualization\n\nThe only runtime visualization entrypoint. It runs one sample with the current tile1024 bootstrap, model-specific topmass, and short-side adaptive stop, then displays the stage trace and initial heatmap detail.

In [ ]:
from pathlib import Path
import os
import sys

def find_repo() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "code" / "top1_hotspot").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the PHF-Crop checkout or one of its subdirectories.")

REPO = find_repo()
os.chdir(REPO)
VIZ_DIR = REPO / "visualization" / "mainline"
for path in (REPO, VIZ_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import phf_crop_visualization_utils as viz

In [ ]:
MODEL = "llava"
DATASET = "vstar"
GPU = "4"
SAMPLE_INDEX = 8
TOPMASS = None  # None uses the model default
LOCALIZATION_STRATEGY = "phf"  # "phf" or "iterative_local"

config = viz.VizConfig(
    repo=REPO,
    model=MODEL,
    task=DATASET,
    gpu=GPU,
    sample_index=SAMPLE_INDEX,
    topmass=TOPMASS,
    initial_heatmap_bootstrap="tile1024",
    adaptive_stop_hotspot_ratio=1.0,
    localization_strategy=LOCALIZATION_STRATEGY,
    save_dir=REPO / "outputs" / "visualization" / "mainline",
)

In [ ]:
context = viz.build_context(config)
adapter, device = viz.load_adapter_for_config(config)
trace, _ = viz.visualize_one(context=context, adapter=adapter, save=False)

print("sample:", trace["row"].get("id"))
print("final crop:", trace["final_bbox"])
print("adaptive stop:", trace.get("adaptive_stop"))
print("original / crop:", trace.get("original_answer"), "/", trace.get("crop_answer"))

# Detail view for the same trace; it does not rerun the model.
tile1024_fig = viz.plot_tile1024_initial_heatmap(trace, context=context, save=False)